# FAD（CLAP music）— Colab GPU

## 依存の前提（確認済み）

| 項目 | 事実 |
|---|---|
| `fadtk==1.1.0` | `Requires-Python: >=3.10,<=3.13` → **3.13.15 は不可**（`<=3.13`＝≤3.13.0） |
| fadtk | `numpy<2`, `scipy<2`, `torch>=2.3,<3` |
| Colab システム | Python 3.13.x / `numpy>=2` → **システム pip に fadtk を入れない** |
| このノート | `uv` で **Python 3.11 venv**（`/content/fad_venv`）を作り、そこでだけ実行 |

定義ファイル: `requirements-fad.txt` / セットアップ: `scripts/setup_fad_venv.py`

## 事前準備
1. ローカルで `fad_wav.zip`（中に `ref_wav/` `gen_wav/`）を Drive `MyDrive/fad_eval/` へ
2. 壊れたランタイムなら **ランタイム → セッションを削除** してから開始
3. GPU ランタイム

In [ ]:
REPO_URL = "https://github.com/Kakeru0307/colab_train.git"
GITHUB_TOKEN = ""  # Private なら PAT（repo 読取）

import os
if GITHUB_TOKEN:
    os.environ["GITHUB_TOKEN"] = GITHUB_TOKEN

def authenticated_repo_url(repo_url: str, token: str) -> str:
    if not token:
        return repo_url
    if repo_url.startswith("https://") and "@" not in repo_url:
        return repo_url.replace("https://", f"https://{token}@")
    return repo_url

CLONE_URL = authenticated_repo_url(REPO_URL, GITHUB_TOKEN)
print("clone:", REPO_URL)
print("token:", "set" if GITHUB_TOKEN else "not set")

WAV_ZIP = "/content/drive/MyDrive/fad_eval/fad_wav.zip"
REF_DIR = ""
GEN_DIR = ""
JSON_OUT = "/content/drive/MyDrive/fad_eval/fad_result.json"

os.environ["FAD_VENV"] = "/content/fad_venv"
FAD_PY = "/content/fad_venv/bin/python"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import shutil
from pathlib import Path

REPO_DIR = Path("repo")

if not Path("scripts/compute_fad.py").exists() and Path("/content").exists():
    os.chdir("/content")

if not Path("scripts/compute_fad.py").exists():
    if REPO_DIR.exists() and not (REPO_DIR / "scripts" / "compute_fad.py").exists():
        print("不完全な repo を削除します")
        shutil.rmtree(REPO_DIR)

    if REPO_DIR.exists() and (REPO_DIR / "scripts" / "compute_fad.py").exists():
        print("既存の repo を使用します")
        os.chdir(REPO_DIR)
    else:
        if not GITHUB_TOKEN:
            print("警告: Private リポジトリの場合、GITHUB_TOKEN を設定してください")
        !git clone {CLONE_URL} repo
        os.chdir("repo")

    if not Path("scripts/compute_fad.py").exists():
        raise FileNotFoundError("clone 失敗。GITHUB_TOKEN かランタイム再起動を確認")

print("git pull ...")
!git pull
print("OK:", os.getcwd())
!git log --oneline -3

In [ ]:
# システム pip には触らない。requirements-fad.txt を 3.11 venv に入れる。
!python scripts/setup_fad_venv.py
assert __import__("pathlib").Path(FAD_PY).is_file(), FAD_PY

In [ ]:
from pathlib import Path
import zipfile
import shutil

WORK = Path("/content/fad_wav")
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)


def _extract_zip_norm(zpath: Path, dest: Path) -> None:
    """Windows zip の backslash パスもディレクトリに展開する."""
    with zipfile.ZipFile(zpath) as zf:
        for info in zf.infolist():
            name = info.filename.replace("\\", "/").lstrip("/")
            if not name or name.endswith("/"):
                if name:
                    (dest / name).mkdir(parents=True, exist_ok=True)
                continue
            out = dest / name
            out.parent.mkdir(parents=True, exist_ok=True)
            with zf.open(info) as src, open(out, "wb") as dst:
                shutil.copyfileobj(src, dst)


ref = Path(REF_DIR) if REF_DIR else None
gen = Path(GEN_DIR) if GEN_DIR else None

if ref and gen and ref.is_dir() and gen.is_dir():
    print("using Drive dirs", ref, gen)
else:
    z = Path(WAV_ZIP)
    assert z.is_file(), f"zip がありません: {z}"
    _extract_zip_norm(z, WORK)
    cands_ref = [p for p in WORK.rglob("ref_wav") if p.is_dir()]
    cands_gen = [p for p in WORK.rglob("gen_wav") if p.is_dir()]
    assert cands_ref and cands_gen, f"zip 内に ref_wav/gen_wav がありません: {list(WORK.iterdir())[:20]}"
    ref, gen = cands_ref[0], cands_gen[0]

n_ref = len(list(ref.glob("*.wav")))
n_gen = len(list(gen.glob("*.wav")))
print(f"ref={ref} ({n_ref})  gen={gen} ({n_gen})")
assert n_ref >= 1 and n_gen >= 1

In [ ]:
from pathlib import Path

out = Path(JSON_OUT)
out.parent.mkdir(parents=True, exist_ok=True)
!{FAD_PY} scripts/compute_fad.py {ref} {gen} --json-out {out}
print(out.read_text(encoding="utf-8"))